# Isolating pruning from fine-tuning, and seed-stability of the collapse

Two supplementary experiments on the CICIoT2023 CNN, using the project's own `src.` modules.

**Experiment 1 — One-shot pruning without fine-tuning.** The compression matrix produces the pruned
cells by magnitude pruning followed by fine-tuning, while the post-training cells (int8, float16)
involve no re-optimisation. To separate the effect of weight removal from the effect of the subsequent
fine-tuning, this experiment prunes the anchor to 80% sparsity with `compression._magnitude_prune`
(the exact prune step used inside `prune_and_finetune`) and evaluates it immediately, before any
fine-tuning. Comparing one-shot pruning, pruning-plus-fine-tuning, and the uncompressed anchor shows
whether the per-class recall collapse is present from pruning alone or emerges only after fine-tuning.

**Experiment 2 — Seed stability of the collapse.** The main results report a single anchor-seed run.
This experiment re-runs the full prune-plus-fine-tune at 80% sparsity across several seeds and records,
per seed, which measurable classes collapse (recall drop exceeding the class's own five-seed 2-sigma
baseline band). It quantifies whether the number of collapsed classes is stable across seeds even when
the exact membership varies.

Collapse is counted on the thirteen measurable classes defined by the manuscript's four-tier scheme; the four unstable/confusable classes that the null-band file groups under "measurable" are excluded, so all counts are directly comparable to the prune80 collapse count reported throughout the paper.

Both experiments write result CSVs under the tables directory via `PATHS`, using the same leakage-aware
split, encoder, scaler, and collapse criterion as the rest of the study. Run top-to-bottom on a GPU
runtime.

In [ ]:
# --- Colab bootstrap (config-driven; never hardcode a path) ---
try:
    from google.colab import drive; drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
except Exception:
    REPO = '.'
import os, sys
os.chdir(REPO); sys.path.insert(0, REPO)

import numpy as np, pandas as pd, torch
from src.config import CFG, PATHS, set_all_seeds
from src import data as D, models as M, compression as C, mitigate as MIT, metrics as MET, train as TR

SEED = CFG['anchor_seed']
set_all_seeds(SEED)
ARCH = 'cnn1d'; DATASET = 'ciciot2023'
print('repo:', REPO)
print('anchor seed:', SEED)

In [ ]:
# >>> RUNTIME CHECK — read before the long cells <<<
# Experiment 2 fine-tunes prune80 at SEVERAL seeds; each finetune is 8 epochs over 2.56M rows.
# On GPU each seed is ~10-20 min; on CPU it is 1-2+ HOURS PER SEED. Use a T4 GPU:
#   Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU, then Restart and run all.
import torch
print('DEVICE:', 'cuda (GPU - good)' if torch.cuda.is_available() else 'cpu (SLOW - switch to T4 GPU)')
assert torch.cuda.is_available(), \
    'CPU runtime detected. Switch to a T4 GPU before running the heavy cells.'

## Load the anchor (M0) and rebuild the exact primary split
Same checkpoint, split, scaler, and encoder used everywhere in the paper. CNN channel widths are inferred directly from the checkpoint tensor shapes (the config does not record them), so the rebuilt model always matches.

In [ ]:
# Load the dataset + the EXACT primary split used everywhere in the paper
df = D.clean(D.load_raw(DATASET, subsample=True, seed=SEED), DATASET)
splits = D.temporal_within_capture_split(df, seed=SEED)
feat_cols = TR.feature_columns(df)

from sklearn.preprocessing import LabelEncoder, StandardScaler
le = LabelEncoder().fit(df['label'].to_numpy())
scaler = StandardScaler().fit(df.loc[splits['train'], feat_cols].to_numpy(np.float32))
n_classes = len(le.classes_)
print('classes:', n_classes, '| train/val/test:',
      len(splits['train']), len(splits['val']), len(splits['test']))

# Load the saved anchor (M0) CNN checkpoint; infer channels from tensor shapes
ckpt_path = PATHS.model(DATASET, ARCH, 'M0', SEED)
ck = torch.load(ckpt_path, map_location='cpu', weights_only=False)
sd = ck['state_dict']
ch1 = int(sd['conv.0.weight'].shape[0]); ch2 = int(sd['conv.3.weight'].shape[0])
arch_kwargs = {'channels': (ch1, ch2)}
print('inferred CNN channels:', arch_kwargs['channels'])

anchor = M.build(ARCH, len(feat_cols), n_classes, **arch_kwargs)
anchor.load_state_dict(sd); anchor = anchor.to(C.DEVICE).eval()
print('anchor loaded.')

## Load the 5-seed baseline null band
This gives the per-class measurable tier and the 2-sigma baseline threshold that defines a genuine collapse (identical criterion to the rest of the paper).

In [ ]:
# The 5-seed baseline null band: per-class mean/std/tier/2-sigma threshold
nb_path = PATHS.tables('baseline', 'cnn1d_M0_null_band_5seed.csv')
null_band = pd.read_csv(nb_path)
band = {r['label']: {'mean': r['mean'], 'twosig': r['null_band_2sigma'], 'tier': r['tier']}
        for _, r in null_band.iterrows()}
# The null-band CSV uses a 3-tier scheme (17 measurable). The paper uses a 4-tier
# scheme: 13 measurable + 4 unstable/confusable. Exclude the 4 unstable classes so
# collapse counts are computed on the paper's 13 measurable set, consistent with
# Table 2, Table 3, and the crux throughout the manuscript.
UNSTABLE = {'DoS-TCP_Flood', 'DDoS-SynonymousIP_Flood', 'DDoS-TCP_Flood', 'DDoS-SYN_Flood'}
measurable = [lbl for lbl, d in band.items()
              if d['tier'] == 'measurable' and lbl not in UNSTABLE]
print(len(measurable), 'measurable classes (paper 4-tier scheme):')
print(measurable)
assert len(measurable) == 13, f'expected 13 measurable, got {len(measurable)}'

def collapsed_classes(recall_by_label):
    out = []
    for lbl in measurable:
        base = band[lbl]['mean']; thr = band[lbl]['twosig']
        r = recall_by_label.get(lbl, float('nan'))
        if not np.isnan(r) and (base - r) > thr:
            out.append(lbl)
    return out

## Experiment 1 — One-shot pruning without fine-tuning

Prune the anchor to 80% sparsity with `compression._magnitude_prune` and evaluate it immediately,
before any fine-tuning. Three conditions are compared on the measurable classes:

* **prune80-oneshot**: pruning alone, no re-optimisation.
* **prune80**: pruning followed by fine-tuning (the compression-matrix cell).
* **M0**: the uncompressed anchor.

If the one-shot condition already collapses the same classes, the collapse follows from weight removal
itself. If the collapse appears only after fine-tuning, it is attributable to the fine-tuning dynamics
rather than to pruning. Per-class test recall is reported for all three conditions.

In [ ]:
# Helper: per-class recall (by label) for a bare model, reusing the project's eval path
@torch.no_grad()
def recall_by_label_for_model(model, is_half=False, is_int8=False):
    entry = {'model': model, 'is_half': is_half, 'is_int8': is_int8}
    rec_idx, macro, _, _, _ = C.evaluate_cell(entry, df, splits, le, scaler, feat_cols, which='test')
    return {le.classes_[ci]: v for ci, v in rec_idx.items()}, macro

# M0 baseline recall (this exact anchor)
rec_M0, mf1_M0 = recall_by_label_for_model(anchor)
print('M0 macro-F1 =', round(mf1_M0, 4))

# Experiment 1: ONE-SHOT prune to 80%, NO fine-tune
set_all_seeds(SEED)
model_oneshot = C._magnitude_prune(anchor, 0.80).to(C.DEVICE).eval()
rec_oneshot, mf1_oneshot = recall_by_label_for_model(model_oneshot)
size_oneshot = C.model_size_report(model_oneshot)
print('prune80-oneshot (NO finetune) macro-F1 =', round(mf1_oneshot, 4),
      '| sparsity =', size_oneshot['sparsity'])

In [ ]:
# Reproduce prune80 WITH fine-tune (same call the pipeline used) for side-by-side
set_all_seeds(SEED)
model_p80, _le2, _sc2 = C.prune_and_finetune(anchor, df, DATASET, splits, SEED, 0.80,
                                             arch=ARCH, verbose=True)
model_p80 = model_p80.to(C.DEVICE).eval()
rec_p80, mf1_p80 = recall_by_label_for_model(model_p80)
print('prune80 (WITH finetune) macro-F1 =', round(mf1_p80, 4))

In [ ]:
# Assemble Experiment 1 table: M0 vs one-shot vs prune80, measurable classes
rows = []
for lbl in measurable:
    r0 = rec_M0.get(lbl, float('nan'))
    ros = rec_oneshot.get(lbl, float('nan'))
    rp8 = rec_p80.get(lbl, float('nan'))
    rows.append({
        'label': lbl,
        'recall_M0': round(r0, 4),
        'recall_oneshot_noft': round(ros, 4),
        'recall_prune80_ft': round(rp8, 4),
        'collapsed_oneshot': (band[lbl]['mean'] - ros) > band[lbl]['twosig'] if not np.isnan(ros) else None,
        'collapsed_prune80': (band[lbl]['mean'] - rp8) > band[lbl]['twosig'] if not np.isnan(rp8) else None,
    })
e1 = pd.DataFrame(rows)

coll_oneshot = collapsed_classes(rec_oneshot)
coll_p80 = collapsed_classes(rec_p80)
print('Collapsed under ONE-SHOT (no finetune):', len(coll_oneshot), 'classes')
print('  ', coll_oneshot)
print('Collapsed under PRUNE80 (with finetune):', len(coll_p80), 'classes')
print('  ', coll_p80)
print('Overlap:', len(set(coll_oneshot) & set(coll_p80)), 'classes collapse under BOTH')
print()
summary1 = pd.DataFrame([
    {'cell': 'M0', 'macro_f1': round(mf1_M0, 4), 'sparsity': 0.0, 'n_collapsed_measurable': 0},
    {'cell': 'prune80_oneshot_noFT', 'macro_f1': round(mf1_oneshot, 4),
     'sparsity': size_oneshot['sparsity'], 'n_collapsed_measurable': len(coll_oneshot)},
    {'cell': 'prune80_FT', 'macro_f1': round(mf1_p80, 4),
     'sparsity': C.model_size_report(model_p80)['sparsity'], 'n_collapsed_measurable': len(coll_p80)}])
print(summary1.to_string(index=False))

out_e1 = PATHS.tables('explain', 'oneshot_vs_finetune_prune80.csv')
e1.to_csv(out_e1, index=False)
out_s1 = PATHS.tables('explain', 'oneshot_vs_finetune_summary.csv')
summary1.to_csv(out_s1, index=False)
print()
print('saved:', out_e1)
print('saved:', out_s1)

## Experiment 2 — Seed stability of the collapse

Re-run prune-plus-fine-tune at 80% sparsity across several seeds and record which measurable classes
collapse under each, using the same 2-sigma baseline criterion as the main analysis. This quantifies
whether the number of collapsed classes is stable across seeds even when the exact set of collapsed
classes varies. Each seed reproduces `compression.prune_and_finetune(..., seed, 0.80)` and is a full
fine-tune (roughly 10-20 minutes on a T4 GPU). `SEEDS` sets the number of runs; three is the minimum,
five gives a tighter estimate.

In [ ]:
# Seeds to run. The anchor seed is already covered above; include it plus new ones.
SEEDS = [SEED, SEED + 1, SEED + 2, SEED + 3, SEED + 4]   # five seeds
print('running prune80 at seeds:', SEEDS)

per_seed_rows = []
per_seed_collapse = {}
per_seed_macro = {}

for s in SEEDS:
    print()
    print('=== seed', s, '===')
    set_all_seeds(s)
    m_s, _l, _sc = C.prune_and_finetune(anchor, df, DATASET, splits, s, 0.80,
                                        arch=ARCH, verbose=False)
    m_s = m_s.to(C.DEVICE).eval()
    rec_s, mf1_s = recall_by_label_for_model(m_s)
    coll_s = collapsed_classes(rec_s)
    per_seed_collapse[s] = coll_s
    per_seed_macro[s] = mf1_s
    print('  macro-F1 =', round(mf1_s, 4), '| collapsed measurable classes:', len(coll_s))
    for lbl in measurable:
        per_seed_rows.append({'seed': s, 'label': lbl,
                              'recall': round(rec_s.get(lbl, float('nan')), 4),
                              'collapsed': lbl in coll_s})
    del m_s; torch.cuda.empty_cache()

In [ ]:
# Summarise Experiment 2: count stability + membership frequency
counts = {s: len(c) for s, c in per_seed_collapse.items()}
print('Collapse COUNT per seed:', counts)
print('ANCHOR seed (seed 0) count:', counts.get(SEED), '<- this is the headline prune80 count')
print('per-seed sequence:', [counts[s] for s in SEEDS])
print('  mean', round(float(np.mean(list(counts.values()))), 1),
      'min', min(counts.values()), 'max', max(counts.values()))
print()

from collections import Counter
freq = Counter()
for s, c in per_seed_collapse.items():
    freq.update(c)
n_seeds = len(SEEDS)
print('Per-class collapse frequency across', n_seeds, 'seeds:')
freq_rows = []
for lbl in measurable:
    k = freq.get(lbl, 0)
    freq_rows.append({'label': lbl, 'times_collapsed': k, 'n_seeds': n_seeds,
                      'fraction': round(k / n_seeds, 3)})
freq_df = pd.DataFrame(freq_rows).sort_values('times_collapsed', ascending=False)
print(freq_df.to_string(index=False))

long_df = pd.DataFrame(per_seed_rows)
out_e2 = PATHS.tables('explain', 'multiseed_prune80_perclass.csv')
long_df.to_csv(out_e2, index=False)
macro_rows = [{'seed': s, 'macro_f1': round(per_seed_macro[s], 4),
               'n_collapsed_measurable': len(per_seed_collapse[s])} for s in SEEDS]
out_s2 = PATHS.tables('explain', 'multiseed_prune80_summary.csv')
pd.DataFrame(macro_rows).to_csv(out_s2, index=False)
out_f2 = PATHS.tables('explain', 'multiseed_prune80_collapse_frequency.csv')
freq_df.to_csv(out_f2, index=False)
print()
print('saved:', out_e2)
print('saved:', out_s2)
print('saved:', out_f2)

## Outputs

The experiments write the following CSVs to the `explain` tables directory:

* `oneshot_vs_finetune_prune80.csv` — per-class test recall under M0, one-shot pruning, and
  pruning-plus-fine-tuning, with per-class collapse flags.
* `oneshot_vs_finetune_summary.csv` — macro-F1, sparsity, and collapsed-class count for the three
  conditions.
* `multiseed_prune80_perclass.csv` — per-class recall and collapse flag for every seed.
* `multiseed_prune80_summary.csv` — macro-F1 and collapsed-class count per seed.
* `multiseed_prune80_collapse_frequency.csv` — how many seeds each measurable class collapses under.

The two quantities of primary interest are the collapsed-class count under one-shot pruning relative to
pruning-plus-fine-tuning (Experiment 1) and the collapsed-class count per seed (Experiment 2).